In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
df_metric = pd.read_csv("metric.csv")
df_trace = pd.read_csv("trace.csv")
df_log = pd.read_csv("log.csv")
df_error = pd.read_csv("error_logs.csv")

# Convert timestamps to UTC datetimes (keep original numeric timestamp column too)
df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)
df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_error['ts'] = pd.to_datetime(df_error['timestamp'], unit='s', utc=True)

# Summaries: row count and min/max timestamp for each file
metric_summary = pd.DataFrame([{
    'rows': df_metric.shape[0],
    'min_ts_utc': df_metric['ts'].min(),
    'max_ts_utc': df_metric['ts'].max()
}])

trace_summary = pd.DataFrame([{
    'rows': df_trace.shape[0],
    'min_ts_utc': df_trace['ts'].min(),
    'max_ts_utc': df_trace['ts'].max()
}])

log_summary = pd.DataFrame([{
    'rows': df_log.shape[0],
    'min_ts_utc': df_log['ts'].min(),
    'max_ts_utc': df_log['ts'].max()
}])

error_summary = pd.DataFrame([{
    'rows': df_error.shape[0],
    'min_ts_utc': df_error['ts'].min(),
    'max_ts_utc': df_error['ts'].max()
}])

# metric: unique (cmdb_id, kpi_name) pairs with counts, top 50
metric_pairs = (
    df_metric.groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
metric_pairs_top = metric_pairs.head(50)

# metric: unique cmdb_id values and unique kpi_name values (as compact series)
metric_unique_cmdb = pd.Series(sorted(df_metric['cmdb_id'].dropna().unique()), name='cmdb_id')
metric_unique_kpi = pd.Series(sorted(df_metric['kpi_name'].dropna().unique()), name='kpi_name')

# trace: unique (cmdb_id, trace_name) pairs with counts, top 50
trace_pairs = (
    df_trace.groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
trace_pairs_top = trace_pairs.head(50)

trace_unique_cmdb = pd.Series(sorted(df_trace['cmdb_id'].dropna().unique()), name='cmdb_id')
trace_unique_name = pd.Series(sorted(df_trace['trace_name'].dropna().unique()), name='trace_name')

# log: unique (cmdb_id, log_name) pairs with counts, top 50
log_pairs = (
    df_log.groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
log_pairs_top = log_pairs.head(50)

log_unique_cmdb = pd.Series(sorted(df_log['cmdb_id'].dropna().unique()), name='cmdb_id')
log_unique_name = pd.Series(sorted(df_log['log_name'].dropna().unique()), name='log_name')

# error_logs: sample up to 20 rows with timestamp (UTC), cmdb_id, message
error_sample = df_error[['ts', 'cmdb_id', 'message']].head(20).reset_index(drop=True)
error_sample = error_sample.rename(columns={'ts': 'timestamp_utc'})

# Display variables (compact)
metric_summary, metric_pairs_top, metric_unique_cmdb, metric_unique_kpi, \
trace_summary, trace_pairs_top, trace_unique_cmdb, trace_unique_name, \
log_summary, log_pairs_top, log_unique_cmdb, log_unique_name, \
error_summary, error_sample

```
Out[1]:
```
summary = (
    "Summary of telemetry files:\n"
    "- metric.csv: 9023 rows; timestamp range 2024-01-24 09:11:00+00:00 to 2024-01-24 09:35:00+00:00.\n"
    "  * 68 unique cmdb_id; 7 unique kpi_name: cpu, diskio, error, latency-50, latency-90, mem, socket, workload.\n"
    "  * Top (cmdb_id, kpi_name) pairs (limited view) show many pairs with count=25 (examples: ts-voucher-service/socket, several ts-admin-basic-info-service KPIs).\n"
    "- trace.csv: 18544 rows; same timestamp range.\n"
    "  * 28 unique cmdb_id; 220 unique trace_name.\n"
    "  * Top trace pairs (limited view) also show repeated 25-row series (including root and many trace.to_* and trace.from_* entries).\n"
    "- log.csv: 2186 rows; same timestamp range.\n"
    "  * 45 unique cmdb_id; 2 unique log_name values: log.error_count, log.row_count.\n"
    "  * Top (cmdb_id, log_name) pairs typically have 24–25 rows (examples: ts-admin-basic-info-service, ts-admin-travel-service).\n"
    "- error_logs.csv: 0 rows (no error log entries found); no sample rows to show.\n\n"
    "Overall: metric, trace, and log telemetry cover the same short time window (09:11–09:35 UTC) with many repeated per-minute series (counts ~24–25 per pair). No error log records were captured in error_logs.csv."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(   rows                min_ts_utc                max_ts_utc
0  9023 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00,                          cmdb_id    kpi_name  count
360           ts-voucher-service      socket     25
0    ts-admin-basic-info-service         cpu     25
1    ts-admin-basic-info-service      diskio     25
2    ts-admin-basic-info-service       error     25
3    ts-admin-basic-info-service  latency-50     25
..                           ...         ...    ...
69               ts-config-mongo      socket     25
70             ts-config-service         cpu     25
71             ts-config-service      diskio     25
72             ts-config-service  latency-50     25
57              ts-basic-service  latency-50     25

[50 rows x 3 columns], 0      ts-admin-basic-info-service
1           ts-admin-order-service
2           ts-admin-route-service
3          ts-admin-travel-service
4            ts-admin-user-service
                  ...             
63                   ts-user-mongo
64                 ts-user-service
65    ts-verification-code-service
66                ts-voucher-mysql
67              ts-voucher-service
Name: cmdb_id, Length: 68, dtype: object, 0           cpu
1        diskio
2         error
3    latency-50
4    latency-90
5           mem
6        socket
7      workload
Name: kpi_name, dtype: object,     rows                min_ts_utc                max_ts_utc
0  18544 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00,                          cmdb_id                                         trace_name  count
7                           root         trace.to_ts-admin-travel-service.row_count     25
728           ts-travel2-service      trace.to_ts-order-other-service.duration_mean     25
729           ts-travel2-service       trace.to_ts-order-other-service.duration_p95     25
730           ts-travel2-service         trace.to_ts-order-other-service.error_rate     25
731           ts-travel2-service          trace.to_ts-order-other-service.row_count     25
..                           ...                                                ...    ...
65   ts-admin-basic-info-service  trace.to_ts-admin-basic-info-service.duration_p95     25
66   ts-admin-basic-info-service    trace.to_ts-admin-basic-info-service.error_rate     25
67   ts-admin-basic-info-service     trace.to_ts-admin-basic-info-service.row_count     25
84       ts-admin-travel-service     trace.to_ts-admin-travel-service.duration_mean     25
85       ts-admin-travel-service      trace.to_ts-admin-travel-service.duration_p95     25

[50 rows x 3 columns], 0                            root
1     ts-admin-basic-info-service
2         ts-admin-travel-service
3            ts-assurance-service
4                 ts-auth-service
                 ...             
23          ts-ticketinfo-service
24               ts-train-service
25              ts-travel-service
26             ts-travel2-service
27                ts-user-service
Name: cmdb_id, Length: 28, dtype: object, 0                          trace.from_root.duration_mean
1                           trace.from_root.duration_p95
2                             trace.from_root.error_rate
3                              trace.from_root.row_count
4      trace.from_ts-admin-basic-info-service.duratio...
                             ...                        
215                trace.to_ts-travel2-service.row_count
216               trace.to_ts-user-service.duration_mean
217                trace.to_ts-user-service.duration_p95
218                  trace.to_ts-user-service.error_rate
219                   trace.to_ts-user-service.row_count
Name: trace_name, Length: 220, dtype: object,    rows                min_ts_utc                max_ts_utc
0  2186 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00,                         cmdb_id         log_name  count
0   ts-admin-basic-info-service  log.error_count     25
1   ts-admin-basic-info-service    log.row_count     25
2       ts-admin-travel-service  log.error_count     25
3       ts-admin-travel-service    log.row_count     25
11              ts-auth-service    log.row_count     25
..                          ...              ...    ...
22            ts-contacts-mongo  log.error_count     24
21           ts-consign-service    log.row_count     24
20           ts-consign-service  log.error_count     24
39        ts-notification-mongo    log.row_count     24
38        ts-notification-mongo  log.error_count     24

[50 rows x 3 columns], 0     ts-admin-basic-info-service
1         ts-admin-travel-service
2              ts-assurance-mongo
3            ts-assurance-service
4                   ts-auth-mongo
                 ...             
40              ts-travel-service
41               ts-travel2-mongo
42             ts-travel2-service
43                ts-ui-dashboard
44                  ts-user-mongo
Name: cmdb_id, Length: 45, dtype: object, 0    log.error_count
1      log.row_count
Name: log_name, dtype: object,    rows min_ts_utc max_ts_utc
0     0        NaT        NaT, Empty DataFrame
Columns: [timestamp_utc, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure timestamp column as UTC datetime exists
if 'ts' not in df_metric.columns or not pd.api.types.is_datetime64tz_dtype(df_metric['ts']):
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Components of interest
components = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Filter metric rows to only these components
df_sub = df_metric[df_metric['cmdb_id'].isin(components)].copy()

# Ensure 'value' is numeric
df_sub['value'] = pd.to_numeric(df_sub['value'], errors='coerce')

# Group and compute required stats
grouped = df_sub.groupby(['cmdb_id', 'kpi_name'], dropna=False)

agg_df = grouped.agg(
    count_of_points = ('value', 'count'),
    min_timestamp_utc = ('ts', 'min'),
    max_timestamp_utc = ('ts', 'max'),
    P5 = ('value', lambda s: np.nanpercentile(s.dropna(), 5) if s.dropna().size>0 else np.nan),
    P50 = ('value', lambda s: np.nanpercentile(s.dropna(), 50) if s.dropna().size>0 else np.nan),
    P90 = ('value', lambda s: np.nanpercentile(s.dropna(), 90) if s.dropna().size>0 else np.nan),
    P95 = ('value', lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan),
    P99 = ('value', lambda s: np.nanpercentile(s.dropna(), 99) if s.dropna().size>0 else np.nan),
    max_value = ('value', 'max')
).reset_index()

# Sort by count desc then cmdb_id, limit to top 100 rows
metrics_summary_filtered_top = agg_df.sort_values(['count_of_points', 'cmdb_id'], ascending=[False, True]).head(100)

# Keep columns order as requested
metrics_summary_filtered_top = metrics_summary_filtered_top[
    ['cmdb_id','kpi_name','count_of_points','min_timestamp_utc','max_timestamp_utc','P5','P50','P90','P95','P99','max_value']
]

metrics_summary_filtered_top

```
Out[2]:
```
Summary of global statistics computed for the five services (no time-window filtering):

- Services analyzed: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service.
- Total (cmdb_id, kpi_name) pairs found: 36.
- Every reported pair has 25 data points (count_of_points = 25).
- Timestamp coverage (global, same for all pairs): min = 2024-01-24 09:11:00+00:00, max = 2024-01-24 09:35:00+00:00.
- For each pair we computed P5, P50 (median), P90, P95, P99, and the maximum value. Results are available in the table (36 rows).

Representative examples (values shown approximate):
- ts-auth-service / cpu: count=25, P5≈13.93, P50≈14.83, P90≈16.45, P95≈16.74, P99≈16.83, max≈16.85.
- ts-auth-service / diskio: P50≈52,601, P95≈62,093, max≈73,937.
- ts-auth-service / latency-50: P50≈0.2576 s, P95≈0.4118 s, max≈0.4243 s.
- ts-travel-service / socket: P50≈15.57, P95≈17.37, max≈18.15.
- ts-travel-service / mem: P50≈266,084,600, P95≈270,238,400, max≈270,378,900.

If you want the full 36-row table printed or exported (sorted by count desc then cmdb_id), I can display it.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count_of_points         min_timestamp_utc         max_timestamp_utc            P5           P50           P90           P95           P99     max_value
0     ts-auth-service         cpu               25 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00  1.392881e+01  1.483259e+01  1.644826e+01  1.674306e+01  1.682859e+01  1.685142e+01
1     ts-auth-service      diskio               25 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00  4.441233e+04  5.260165e+04  6.052561e+04  6.209321e+04  7.113558e+04  7.393728e+04
2     ts-auth-service  latency-50               25 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00  2.177043e-01  2.576028e-01  3.487742e-01  4.117970e-01  4.242045e-01  4.243308e-01
3     ts-auth-service  latency-90               25 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00  5.810197e-01  7.934438e-01  1.150892e+00  1.374300e+00  1.682879e+00  1.763095e+00
4     ts-auth-service         mem               25 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00  2.245598e+08  2.360216e+08  2.415030e+08  2.417483e+08  2.418082e+08  2.418212e+08
..                ...         ...              ...                       ...                       ...           ...           ...           ...           ...           ...           ...
31  ts-travel-service  latency-50               25 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00  1.269612e-02  1.997521e-02  2.931172e-02  5.061257e-02  8.346281e-02  9.237621e-02
32  ts-travel-service  latency-90               25 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00  1.224544e-01  2.591511e-01  4.085010e-01  5.206471e-01  8.647623e-01  9.651582e-01
33  ts-travel-service         mem               25 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00  2.532723e+08  2.660846e+08  2.701401e+08  2.702384e+08  2.703489e+08  2.703789e+08
34  ts-travel-service      socket               25 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00  1.403667e+01  1.556667e+01  1.687000e+01  1.737000e+01  1.799000e+01  1.815000e+01
35  ts-travel-service    workload               25 2024-01-24 09:11:00+00:00 2024-01-24 09:35:00+00:00  7.361827e+00  8.798883e+00  1.017302e+01  1.065810e+01  1.090192e+01  1.094223e+01

[36 rows x 11 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing df_metric if present; otherwise load
try:
    df_metric
except NameError:
    df_metric = pd.read_csv("metric.csv")
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure ts column exists and is timezone-aware UTC
if 'ts' not in df_metric.columns or not pd.api.types.is_datetime64tz_dtype(df_metric['ts']):
    df_metric['ts'] = pd.to_datetime(df_metric['timestamp'], unit='s', utc=True)

# Ensure value is numeric
df_metric['value'] = pd.to_numeric(df_metric['value'], errors='coerce')

# Components of interest
components = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# 1) Compute global P95 per (cmdb_id, kpi_name) for these components using full data
df_comp = df_metric[df_metric['cmdb_id'].isin(components)].copy()

# Group and compute P95 (global)
global_p95 = (
    df_comp.groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan)
    .reset_index()
    .rename(columns={'value': 'P95_value'})
)

# 2) Filter metric.csv to incident window (inclusive)
window_start = pd.to_datetime("2024-01-24 09:08:53", utc=True)
window_end   = pd.to_datetime("2024-01-24 09:38:53", utc=True)

df_window = df_metric[
    (df_metric['ts'] >= window_start) &
    (df_metric['ts'] <= window_end) &
    (df_metric['cmdb_id'].isin(components))
].copy()

# 3) For each (cmdb_id,kpi_name) find points in window where value >= global P95
# Merge P95 onto window rows
df_win_with_p95 = df_window.merge(global_p95, on=['cmdb_id','kpi_name'], how='left')

# Mark anomalies where value >= P95_value (handle NaN P95 by treating as no anomaly)
df_win_with_p95['is_anomaly'] = False
mask = df_win_with_p95['P95_value'].notna()
df_win_with_p95.loc[mask, 'is_anomaly'] = df_win_with_p95.loc[mask, 'value'] >= df_win_with_p95.loc[mask, 'P95_value']

# Aggregate anomalies by pair
anoms = (
    df_win_with_p95[df_win_with_p95['is_anomaly']]
    .groupby(['cmdb_id','kpi_name'], dropna=False)
    .agg(
        count_of_anomalies_in_window = ('value','count'),
        earliest_anomaly_timestamp_utc = ('ts','min'),
        latest_anomaly_timestamp_utc = ('ts','max'),
        max_value_in_window = ('value','max'),
        P95_value = ('P95_value','first')  # same for the group
    )
    .reset_index()
)

# Compute severity = max_value_in_window / P95_value, handle division by zero/NaN
anoms['severity'] = anoms.apply(
    lambda r: (r['max_value_in_window'] / r['P95_value']) if pd.notna(r['P95_value']) and r['P95_value'] != 0 else np.nan,
    axis=1
)

# 4) Keep rows with count > 0 only, sort by count desc then severity desc, limit top 20
if anoms.shape[0] == 0:
    result = "NO_ANOMALIES_FOUND"
else:
    result = (
        anoms
        .sort_values(['count_of_anomalies_in_window','severity'], ascending=[False, False])
        .loc[:, ['cmdb_id','kpi_name','count_of_anomalies_in_window',
                 'earliest_anomaly_timestamp_utc','latest_anomaly_timestamp_utc',
                 'max_value_in_window','P95_value','severity']]
        .head(20)
        .reset_index(drop=True)
    )

# Display the result (variable name)
result

```
Out[3]:
```
Summary of anomaly scan (metric.csv) for the five services, using global P95 (full history) and inspecting the incident window 2024-01-24 09:08:53 → 2024-01-24 09:38:53 UTC:

- Total pairs with anomalies in the window: 20 (returned top 20 rows).
- Most anomalies occurred between 2024-01-24 09:11:00 and 2024-01-24 09:35:00 UTC.

Top findings (concise highlights from the result):
- ts-travel-service / error: count=25, earliest=09:11, latest=09:35, max=0, P95=0 → severity is NaN. Interpretation: P95 is zero and metric values are zero, so this "anomaly" is not informative (baseline=0).
- ts-route-service / diskio: count=2, earliest=09:12, latest=09:35, max≈1.84e7, P95≈62,668.69, severity≈293.87 — very large spike in disk I/O; highest severity in the table.
- ts-train-service / socket: count=3, earliest=09:11, latest=09:34, max=13.10, P95=13.00, severity≈1.0077 (small exceedances).
- ts-train-service / latency-90: count=2, earliest=09:11, latest=09:15, max≈0.070999, P95≈0.024155, severity≈2.94 (notable latency increase).
- ts-route-service / latency-50: count=2, earliest=09:24, latest=09:25, max≈0.0411089, P95≈0.021469, severity≈1.91.
- ts-travel-service / latency-90 and latency-50: each count=2, severities ≈1.85 and ≈1.83 respectively (small-to-moderate latency increases).
- ts-train-service / cpu and ts-travel-service / cpu: each count=2, severities ≈1.48 and ≈1.45 respectively (CPU above P95 but moderate).
- Several other latency/cpu/socket/workload pairs for ts-auth, ts-order, ts-route, ts-train, ts-travel appear with count=2 and severities in the ~1.05–1.33 range.

Overall interpretation and recommended next steps:
- The most urgent signal is the huge diskio spike on ts-route-service (severity ≫ 1). Investigate disk I/O on ts-route-service immediately (processes, disk subsystem, queries).
- Significant latency increases appear on ts-train and ts-travel services (latency-90 and latency-50). Correlate with traces/logs around 09:11–09:15 and 09:24–09:25.
- Ignore the ts-travel-service/error series: P95=0 and values=0 make that row non-actionable.
- For the moderate CPU/socket/latency exceedances (severity ~1.1–1.5), review traces and logs for those services to determine impact.

If you want, I can print the full 20-row anomaly table again or show the specific raw metric points for any of the highlighted pairs.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count_of_anomalies_in_window earliest_anomaly_timestamp_utc latest_anomaly_timestamp_utc  max_value_in_window     P95_value    severity
0   ts-travel-service       error                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00         0.000000e+00      0.000000         NaN
1    ts-train-service      socket                             3      2024-01-24 09:11:00+00:00    2024-01-24 09:34:00+00:00         1.310000e+01     13.000000    1.007692
2    ts-route-service      diskio                             2      2024-01-24 09:12:00+00:00    2024-01-24 09:35:00+00:00         1.841662e+07  62668.688180  293.872720
3    ts-train-service  latency-90                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:15:00+00:00         7.099892e-02      0.024155    2.939307
4    ts-route-service  latency-50                             2      2024-01-24 09:24:00+00:00    2024-01-24 09:25:00+00:00         4.110789e-02      0.021469    1.914722
5   ts-travel-service  latency-90                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00         9.651582e-01      0.520647    1.853766
6   ts-travel-service  latency-50                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00         9.237621e-02      0.050613    1.825163
7    ts-train-service  latency-50                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00         1.385246e-02      0.008482    1.633143
8    ts-train-service         cpu                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00         1.727532e+01     11.651856    1.482624
9   ts-travel-service         cpu                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00         2.500272e+01     17.253110    1.449172
10    ts-auth-service  latency-90                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:24:00+00:00         1.763095e+00      1.374300    1.282905
11   ts-order-service  latency-90                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00         8.926694e-02      0.071334    1.251389
12   ts-order-service  latency-50                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00         1.980530e-02      0.016257    1.218251
13    ts-auth-service      diskio                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:21:00+00:00         7.393728e+04  62093.207666    1.190747
14   ts-order-service    workload                             2      2024-01-24 09:13:00+00:00    2024-01-24 09:15:00+00:00         7.937750e+00      7.152992    1.109710
15   ts-route-service  latency-90                             2      2024-01-24 09:25:00+00:00    2024-01-24 09:26:00+00:00         8.951895e-02      0.082522    1.084788
16   ts-order-service         cpu                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00         1.215539e+01     11.365695    1.069480
17  ts-travel-service      diskio                             2      2024-01-24 09:25:00+00:00    2024-01-24 09:28:00+00:00         6.456601e+04  60453.832216    1.068022
18   ts-order-service      socket                             2      2024-01-24 09:25:00+00:00    2024-01-24 09:29:00+00:00         1.186667e+01     11.270000    1.052943
19    ts-auth-service      socket                             2      2024-01-24 09:14:00+00:00    2024-01-24 09:27:00+00:00         1.875000e+01     17.893333    1.047876```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse existing df_trace if available, otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure timestamp parsed as UTC datetime tz-aware (unix seconds)
df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure value is numeric
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Components of interest
components = ["ts-route-service","ts-train-service","ts-travel-service","ts-auth-service","ts-order-service"]

# Restrict dataset to those components for analysis
df_trace_sub = df_trace[df_trace['cmdb_id'].isin(components)].copy()

# 1) Compute global P95 per (cmdb_id, trace_name) using full available data (no time filtering)
global_p95_trace = (
    df_trace_sub
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan)
    .reset_index()
    .rename(columns={'value':'P95_value'})
)

# 2) Filter trace.csv to the incident window (inclusive)
window_start = pd.to_datetime("2024-01-24 09:08:53", utc=True)
window_end   = pd.to_datetime("2024-01-24 09:38:53", utc=True)

df_trace_window = df_trace_sub[
    (df_trace_sub['ts'] >= window_start) &
    (df_trace_sub['ts'] <= window_end)
].copy()

# 3) Find rows in the window where value >= global P95 for the corresponding pair
df_tw = df_trace_window.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')

# Mark anomalies: only where P95 exists (notna) and value >= P95_value
mask_p95 = df_tw['P95_value'].notna()
df_tw['is_anomaly'] = False
df_tw.loc[mask_p95, 'is_anomaly'] = df_tw.loc[mask_p95, 'value'] >= df_tw.loc[mask_p95, 'P95_value']

# Aggregate anomaly info per (cmdb_id, trace_name)
anoms_trace = (
    df_tw[df_tw['is_anomaly']]
    .groupby(['cmdb_id','trace_name'], dropna=False)
    .agg(
        count_of_anomalies_in_window = ('value','count'),
        earliest_anomaly_timestamp_utc = ('ts','min'),
        latest_anomaly_timestamp_utc = ('ts','max'),
        max_value_in_window = ('value','max'),
        P95_value = ('P95_value','first')
    )
    .reset_index()
)

# Compute severity (handle P95==0 or NaN)
def compute_sev(row):
    p = row['P95_value']
    m = row['max_value_in_window']
    if pd.isna(p) or p == 0:
        return np.nan
    return m / p

if not anoms_trace.empty:
    anoms_trace['severity'] = anoms_trace.apply(compute_sev, axis=1)

# 4) Filter rows with count>0 (already true), sort by count desc then severity desc, limit 20
if anoms_trace.shape[0] == 0:
    result = "NO_ANOMALIES_FOUND"
else:
    result = (
        anoms_trace
        .sort_values(['count_of_anomalies_in_window','severity'], ascending=[False, False])
        .loc[:, ['cmdb_id','trace_name','count_of_anomalies_in_window',
                 'earliest_anomaly_timestamp_utc','latest_anomaly_timestamp_utc',
                 'max_value_in_window','P95_value','severity']]
        .head(20)
        .reset_index(drop=True)
    )

result

```
Out[4]:
```
Summary of trace analysis for components [ts-route-service, ts-train-service, ts-travel-service, ts-auth-service, ts-order-service] in the incident window 2024-01-24 09:08:53 → 2024-01-24 09:38:53 UTC:

- We computed global P95 per (cmdb_id, trace_name) using full data and then flagged window points >= that P95.
- The flagged traces are all error-rate series (many pairs), but every reported P95_value = 0 and max_value_in_window = 0, so severity is NaN.
- In other words: there are no meaningful trace error-rate anomalies (error rates are zero and baseline P95 is zero) in the incident window for these services.
- Conclusion: trace error_rate telemetry does not indicate actionable anomalies for these services during the window. Recommend checking non-error trace features (duration/latency) or the metric/log anomalies found earlier for root-cause signals.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                       trace_name  count_of_anomalies_in_window earliest_anomaly_timestamp_utc latest_anomaly_timestamp_utc  max_value_in_window  P95_value  severity
0    ts-auth-service                       trace.from_root.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
1    ts-auth-service            trace.from_ts-auth-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
2    ts-auth-service              trace.to_ts-auth-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
3   ts-order-service                       trace.from_root.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
4   ts-order-service  trace.from_ts-inside-payment-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
5   ts-order-service           trace.from_ts-order-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
6   ts-order-service            trace.from_ts-seat-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
7   ts-order-service          trace.from_ts-travel-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
8   ts-order-service             trace.to_ts-order-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
9   ts-order-service           trace.to_ts-station-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
10  ts-route-service           trace.from_ts-basic-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
11  ts-route-service           trace.from_ts-route-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
12  ts-route-service          trace.from_ts-travel-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
13  ts-route-service         trace.from_ts-travel2-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
14  ts-route-service             trace.to_ts-route-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
15  ts-train-service           trace.from_ts-basic-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
16  ts-train-service           trace.from_ts-train-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
17  ts-train-service          trace.from_ts-travel-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
18  ts-train-service         trace.from_ts-travel2-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN
19  ts-train-service             trace.to_ts-train-service.error_rate                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                  0.0        0.0       NaN```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse df_trace if available; otherwise load
try:
    df_trace
except NameError:
    df_trace = pd.read_csv("trace.csv")

# Ensure ts column exists and is UTC tz-aware
if 'ts' not in df_trace.columns or not pd.api.types.is_datetime64tz_dtype(df_trace['ts']):
    df_trace['ts'] = pd.to_datetime(df_trace['timestamp'], unit='s', utc=True)

# Ensure numeric value
df_trace['value'] = pd.to_numeric(df_trace['value'], errors='coerce')

# Components and duration-related trace_name filter
components = ["ts-route-service","ts-train-service","ts-travel-service","ts-auth-service","ts-order-service"]
duration_pattern = r'duration'  # covers duration, duration_mean, duration_p95, etc.

# Restrict to components and duration-related trace_names
df_trace_sub = df_trace[df_trace['cmdb_id'].isin(components)].copy()
df_trace_sub = df_trace_sub[df_trace_sub['trace_name'].str.contains(duration_pattern, case=False, na=False)].copy()

# 1) Compute global P95 per (cmdb_id, trace_name) using full available data (no time filtering)
global_p95_trace = (
    df_trace_sub
    .groupby(['cmdb_id','trace_name'], dropna=False)['value']
    .agg(lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan)
    .reset_index()
    .rename(columns={'value':'P95_value'})
)

# 2) Filter to incident window (inclusive)
window_start = pd.to_datetime("2024-01-24 09:08:53", utc=True)
window_end   = pd.to_datetime("2024-01-24 09:38:53", utc=True)

df_trace_window = df_trace_sub[
    (df_trace_sub['ts'] >= window_start) &
    (df_trace_sub['ts'] <= window_end)
].copy()

# 3) Merge P95 onto window rows and flag anomalies where value >= P95 (only when P95 is not NaN)
df_tw = df_trace_window.merge(global_p95_trace, on=['cmdb_id','trace_name'], how='left')

mask_p95 = df_tw['P95_value'].notna()
df_tw['is_anomaly'] = False
df_tw.loc[mask_p95, 'is_anomaly'] = df_tw.loc[mask_p95, 'value'] >= df_tw.loc[mask_p95, 'P95_value']

# Aggregate anomalies per pair
anoms_trace_duration = (
    df_tw[df_tw['is_anomaly']]
    .groupby(['cmdb_id','trace_name'], dropna=False)
    .agg(
        count_of_anomalies_in_window = ('value','count'),
        earliest_anomaly_timestamp_utc = ('ts','min'),
        latest_anomaly_timestamp_utc = ('ts','max'),
        max_value_in_window = ('value','max'),
        P95_value = ('P95_value','first')
    )
    .reset_index()
)

# Compute severity safely
def compute_sev(row):
    p = row['P95_value']
    m = row['max_value_in_window']
    if pd.isna(p) or p == 0:
        return np.nan
    return m / p

if not anoms_trace_duration.empty:
    anoms_trace_duration['severity'] = anoms_trace_duration.apply(compute_sev, axis=1)

# 4) Keep rows with count>0 (already true), sort by count desc then severity desc, limit top 20
if anoms_trace_duration.shape[0] == 0:
    result = "NO_ANOMALIES_FOUND"
else:
    result = (
        anoms_trace_duration
        .sort_values(['count_of_anomalies_in_window','severity'], ascending=[False, False])
        .loc[:, ['cmdb_id','trace_name','count_of_anomalies_in_window',
                 'earliest_anomaly_timestamp_utc','latest_anomaly_timestamp_utc',
                 'max_value_in_window','P95_value','severity']]
        .head(20)
        .reset_index(drop=True)
    )

result

```
Out[5]:
```
Summary of duration-trace anomalies (components: ts-route, ts-train, ts-travel, ts-auth, ts-order) in the incident window 2024-01-24 09:08:53 → 09:38:53 UTC:

- Total (cmdb_id, trace_name) pairs with duration anomalies: 20.  
- Every reported pair had 2 anomalous points in the window (count_of_anomalies_in_window = 2). Timestamps largely fall between ~09:11 and ~09:28 UTC.

Top anomaly highlights (sorted by severity):
1. ts-order-service / trace.from_ts-security-service.duration_p95 — count=2, window max=0.052905, P95=0.014419, severity≈3.67 (largest relative spike).
2. ts-order-service / trace.from_ts-security-service.duration_mean — count=2, max=0.023896, P95=0.009643, severity≈2.48.
3. ts-train-service / trace.from_ts-travel-service.duration_p95 — count=2, max=0.069107, P95=0.033948, severity≈2.04.
4. ts-travel-service / trace.to_ts-train-service.duration_p95 — count=2, max=0.069107, P95=0.033948, severity≈2.04.
5. ts-order-service / trace.from_ts-inside-payment-service.duration_* — count=2, max≈0.04530, P95≈0.02448, severity≈1.85.
6. ts-train-service / trace.from_ts-basic-service.duration_p95 — count=2, severity≈1.83.
7. ts-auth-service / trace.from/to_ts-auth-service.duration_p95 — count=2 each, severity≈1.56.
8. Additional affected flows include seat-service, order-service internal flows, route-service duration traces, admin-travel-service durations, and preserve-service durations — severities mostly between ~1.33 and ~1.50.

Interpretation and recommended next steps:
- The strongest relative spike is in traces where ts-order-service calls the security service (duration_p95 and duration_mean). This suggests the security-service interaction slowed significantly and may be a primary contributor to request latency.
- Related increases appear in travel→train and train→basic flows, and in seat/payment related traces — indicating cross-service latency propagation.
- Action items: investigate ts-order → security-service call path (service logs, recent deploys, thread/pool saturation, DB or downstream latency), and correlate with the metric anomalies previously found (diskio spike on ts-route-service and latency/cpu signals on ts-train/ts-travel). Also inspect traces around ~09:11–09:15 and ~09:24–09:25 UTC where many anomalies occurred.
- If you want, I can print the full 20-row table again or fetch the raw trace points for any highlighted pair.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                         trace_name  count_of_anomalies_in_window earliest_anomaly_timestamp_utc latest_anomaly_timestamp_utc  max_value_in_window  P95_value  severity
0    ts-order-service        trace.from_ts-security-service.duration_p95                             2      2024-01-24 09:12:00+00:00    2024-01-24 09:13:00+00:00             0.052905   0.014419  3.669098
1    ts-order-service       trace.from_ts-security-service.duration_mean                             2      2024-01-24 09:12:00+00:00    2024-01-24 09:21:00+00:00             0.023896   0.009643  2.478061
2    ts-train-service          trace.from_ts-travel-service.duration_p95                             2      2024-01-24 09:12:00+00:00    2024-01-24 09:15:00+00:00             0.069107   0.033948  2.035683
3   ts-travel-service             trace.to_ts-train-service.duration_p95                             2      2024-01-24 09:12:00+00:00    2024-01-24 09:15:00+00:00             0.069107   0.033948  2.035683
4    ts-order-service  trace.from_ts-inside-payment-service.duration_...                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:13:00+00:00             0.045298   0.024480  1.850431
5    ts-train-service           trace.from_ts-basic-service.duration_p95                             2      2024-01-24 09:15:00+00:00    2024-01-24 09:28:00+00:00             0.050138   0.027420  1.828493
6     ts-auth-service            trace.from_ts-auth-service.duration_p95                             2      2024-01-24 09:12:00+00:00    2024-01-24 09:13:00+00:00             0.006598   0.004216  1.564968
7     ts-auth-service              trace.to_ts-auth-service.duration_p95                             2      2024-01-24 09:12:00+00:00    2024-01-24 09:13:00+00:00             0.006598   0.004216  1.564968
8   ts-travel-service             trace.to_ts-seat-service.duration_mean                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00             0.205169   0.137153  1.495910
9   ts-travel-service           trace.from_ts-seat-service.duration_mean                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00             0.055985   0.037890  1.477592
10   ts-order-service           trace.from_ts-seat-service.duration_mean                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00             0.029656   0.020353  1.457119
11   ts-order-service           trace.from_ts-order-service.duration_p95                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00             0.016383   0.011287  1.451437
12   ts-order-service             trace.to_ts-order-service.duration_p95                             2      2024-01-24 09:11:00+00:00    2024-01-24 09:12:00+00:00             0.016383   0.011287  1.451437
13  ts-travel-service   trace.from_ts-admin-travel-service.duration_mean                             2      2024-01-24 09:24:00+00:00    2024-01-24 09:25:00+00:00             0.395499   0.273501  1.446061
14   ts-route-service           trace.from_ts-route-service.duration_p95                             2      2024-01-24 09:24:00+00:00    2024-01-24 09:25:00+00:00             0.072619   0.051928  1.398459
15   ts-route-service             trace.to_ts-route-service.duration_p95                             2      2024-01-24 09:24:00+00:00    2024-01-24 09:25:00+00:00             0.072619   0.051928  1.398459
16  ts-travel-service        trace.from_ts-preserve-service.duration_p95                             2      2024-01-24 09:12:00+00:00    2024-01-24 09:16:00+00:00             1.255621   0.901115  1.393409
17    ts-auth-service                       trace.from_root.duration_p95                             2      2024-01-24 09:13:00+00:00    2024-01-24 09:24:00+00:00             1.739941   1.286973  1.351964
18   ts-route-service          trace.from_ts-route-service.duration_mean                             2      2024-01-24 09:24:00+00:00    2024-01-24 09:25:00+00:00             0.007558   0.005675  1.331660
19   ts-route-service            trace.to_ts-route-service.duration_mean                             2      2024-01-24 09:24:00+00:00    2024-01-24 09:25:00+00:00             0.007558   0.005675  1.331660```
```

In [ ]:
# In[6]:

import pandas as pd
import numpy as np

# Reuse df_log if present, otherwise load
try:
    df_log
except NameError:
    df_log = pd.read_csv("log.csv")

# Ensure timestamp parsed as UTC datetime (unix seconds) and value numeric
df_log['ts'] = pd.to_datetime(df_log['timestamp'], unit='s', utc=True)
df_log['value'] = pd.to_numeric(df_log['value'], errors='coerce')

# Components of interest
components = ["ts-auth-service","ts-order-service","ts-route-service","ts-train-service","ts-travel-service"]

# Restrict to those components for threshold calculation and later filtering
df_log_sub_all = df_log[df_log['cmdb_id'].isin(components)].copy()

# 1) Compute global P95 per (cmdb_id, log_name) using full available data (no time filtering)
global_p95_log = (
    df_log_sub_all
    .groupby(['cmdb_id','log_name'], dropna=False)['value']
    .agg(lambda s: np.nanpercentile(s.dropna(), 95) if s.dropna().size>0 else np.nan)
    .reset_index()
    .rename(columns={'value':'P95_value'})
)

# 2) Filter log rows to the incident window (inclusive)
window_start = pd.to_datetime("2024-01-24 09:08:53", utc=True)
window_end   = pd.to_datetime("2024-01-24 09:38:53", utc=True)

df_log_window = df_log_sub_all[
    (df_log_sub_all['ts'] >= window_start) &
    (df_log_sub_all['ts'] <= window_end)
].copy()

# 3) Merge P95 onto window rows and find rows where value >= P95_value (only when P95 exists)
df_lw = df_log_window.merge(global_p95_log, on=['cmdb_id','log_name'], how='left')

mask_p95 = df_lw['P95_value'].notna()
df_lw['is_anomaly'] = False
df_lw.loc[mask_p95, 'is_anomaly'] = df_lw.loc[mask_p95, 'value'] >= df_lw.loc[mask_p95, 'P95_value']

# Aggregate anomalies by pair
anoms_log = (
    df_lw[df_lw['is_anomaly']]
    .groupby(['cmdb_id','log_name'], dropna=False)
    .agg(
        count_of_anomalies_in_window = ('value','count'),
        earliest_anomaly_timestamp_utc = ('ts','min'),
        latest_anomaly_timestamp_utc = ('ts','max'),
        max_value_in_window = ('value','max'),
        P95_value = ('P95_value','first')
    )
    .reset_index()
)

# Compute severity safely (handle P95 == 0 or NaN)
def compute_severity(row):
    p = row['P95_value']
    m = row['max_value_in_window']
    if pd.isna(p) or p == 0:
        return np.nan
    return m / p

if not anoms_log.empty:
    anoms_log['severity'] = anoms_log.apply(compute_severity, axis=1)

# 4) Filter pairs with count>0 (already) and sort by count desc then severity desc, limit to top 20
if anoms_log.shape[0] == 0:
    result = "NO_LOG_ANOMALIES_FOUND"
else:
    result = (
        anoms_log
        .sort_values(['count_of_anomalies_in_window','severity'], ascending=[False, False])
        .loc[:, ['cmdb_id','log_name','count_of_anomalies_in_window',
                 'earliest_anomaly_timestamp_utc','latest_anomaly_timestamp_utc',
                 'max_value_in_window','P95_value','severity']]
        .head(20)
        .reset_index(drop=True)
    )

# Display result (compact)
result

```
Out[6]:
```
Summary of log analysis for the five services in the incident window (2024-01-24 09:08:53 → 09:38:53 UTC):

- log.error_count:
  - ts-auth, ts-order, ts-route, ts-travel: flagged rows show count=25 in-window but max=0 and P95=0 → non-actionable (no errors recorded; baseline P95 is zero).

- log.row_count (actionable small spikes):
  - ts-order-service: count_of_anomalies_in_window=2, earliest=2024-01-24 09:13:00, latest=09:28:00, max=486, P95=470.2, severity≈1.034.
  - ts-travel-service: count=2, earliest=09:13:00, latest=09:28:00, max=2091, P95=2028.0, severity≈1.031.
  - ts-auth-service: count=2, earliest=09:21:00, latest=09:34:00, max=356, P95=346.4, severity≈1.028.
  - ts-route-service: count=2, earliest=09:15:00, latest=09:29:00, max=1069, P95=1049.4, severity≈1.019.

Interpretation and next steps:
- The error_count series are zeros and not informative here.
- Several services show modest increases in log.row_count (≈1.02–1.03× above their global P95) at times that overlap the metric/trace anomalies previously found.
- Recommend correlating these small log-volume spikes with the metric (disk I/O spike on ts-route-service) and the trace duration anomalies (notably ts-order → security-service and travel/train flows) to find causal links. Check service logs around the listed timestamps for more context.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id         log_name  count_of_anomalies_in_window earliest_anomaly_timestamp_utc latest_anomaly_timestamp_utc  max_value_in_window  P95_value  severity
0    ts-auth-service  log.error_count                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                    0        0.0       NaN
1   ts-order-service  log.error_count                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                    0        0.0       NaN
2   ts-route-service  log.error_count                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                    0        0.0       NaN
3  ts-travel-service  log.error_count                            25      2024-01-24 09:11:00+00:00    2024-01-24 09:35:00+00:00                    0        0.0       NaN
4   ts-order-service    log.row_count                             2      2024-01-24 09:13:00+00:00    2024-01-24 09:28:00+00:00                  486      470.2  1.033603
5  ts-travel-service    log.row_count                             2      2024-01-24 09:13:00+00:00    2024-01-24 09:28:00+00:00                 2091     2028.0  1.031065
6    ts-auth-service    log.row_count                             2      2024-01-24 09:21:00+00:00    2024-01-24 09:34:00+00:00                  356      346.4  1.027714
7   ts-route-service    log.row_count                             2      2024-01-24 09:15:00+00:00    2024-01-24 09:29:00+00:00                 1069     1049.4  1.018677```
```